In [1]:
import sys
import os

sys.path.append('..')

import torch
from cying import nn as cynn
from datasets import load_dataset
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, IterableDataset

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

from torchinfo import summary
num_class = 2*10**4
opt_lang_model = cynn.LangModel(
    vocab_size = num_class,
    d_model = 256,
    num_heads = 8,
    hidden_width = 512,
    num_layers = 1,
    padding_idx = 0
).to(device)
summary(opt_lang_model)

Layer (type:depth-idx)                   Param #
LangModel                                512
├─Embedding: 1-1                         5,120,000
├─ModuleList: 1-2                        --
│    └─LangLayer: 2-1                    2,048
│    │    └─Sequential: 3-1              65,792
│    │    └─Sequential: 3-2              65,792
│    │    └─Linear: 3-3                  65,536
│    │    └─Linear: 3-4                  65,536
│    │    └─Sequential: 3-5              262,912
│    │    └─RMSNorm: 3-6                 --
├─RMSNorm: 1-3                           --
├─Linear: 1-4                            5,140,000
Total params: 10,788,128
Trainable params: 10,788,128
Non-trainable params: 0

In [3]:
import json
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader

scale = 1
max_seq_len = 128
batch_size = 1
data_path = f'/root/autodl-tmp/data'
tokenizer_path = f"./cci3_tokenizer.json"


class CCI3(IterableDataset):
    def __init__(self, data_path, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.dataset = load_dataset(
            data_path,
            split="train",
            streaming=True,
        ).shuffle(seed=42, buffer_size=100)

    def __iter__(self):
        for sample in self.dataset:
            encoding = self.tokenizer.encode(sample["text"])

            input_ids = torch.tensor(encoding.ids, dtype=torch.long)
            attention_mask = torch.tensor(
                encoding.attention_mask, dtype=torch.long
            )
            labels = input_ids.clone()

            yield input_ids[:-1], labels[1:]


tokenizer = Tokenizer.from_file(tokenizer_path)
tokenizer.enable_truncation(max_length=max_seq_len)

tokenizer.enable_padding(
    length=max_seq_len,
    pad_id=0
)

dataset = CCI3(
    data_path,
    tokenizer
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size
)

Resolving data files:   0%|          | 0/672 [00:00<?, ?it/s]

In [4]:
for x, y in dataloader:
    print(x, y)
    break

tensor([[ 1916,  2277,  3193,   860,   265,  1817,   653,  9980,    15,   503,
           426,   699,  1822,  8483,  1795,  1841,   263,  2513,  5650,    15,
          1171, 18504,   426,   699,  6083,  6117,  3193,   860,   265,   399,
          5773,  2047,   332,  1481,    15,   282,   677,  1677,  1263,   775,
          3781,   970,  2271,  2444,  9029,   265,   423,   864,   700,   566,
           407,    15,   300,  8944,  1539,  2179,  8461,   317,    15, 12441,
          9198,   676,   407,  1127,  4062,   686,   851,    29,  1320,  3995,
           332,    46, 19783,    44,    15, 10407,   999,  1036,   362,  2269,
          1681,  7501,   389,  1782,    15,  2223,  2506,  3692,  7373,    22,
            17,    26,  1782,   265,   949,  1516,   281,   738,   593,   728,
         16435,  1538,   820,  3118,    15,  1783,   614,   423,   807, 16479,
          8780,   202, 10305,  4636,    15,  5773,  2047,   332,  1481,   263,
          7432,   562,   851,  1247, 14028, 19619,  

In [ ]:
from tqdm import tqdm

epoch = 10**2
lr = 1e-3
optim = torch.optim.Adam(opt_lang_model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

for i in range(epoch):
    pbar = tqdm(dataloader)
    k = 1
    opt_lang_model.train()
    for token_ids, tgt_ids in pbar:
        token_ids = token_ids.to(device)
        tgt_ids = tgt_ids.to(device)
        outputs = opt_lang_model(
            token_seq=token_ids,
            causal_mask=torch.triu(torch.ones(max_seq_len-1, max_seq_len-1), diagonal=1).bool().to(device)
        )
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            tgt_ids.reshape(-1)
        ) / scale
        loss.backward()
        if k % scale == 0:
            k = 1
            optim.step()
            optim.zero_grad()
        else:
            k += 1
        pbar.set_description(f"Epoch {i+1}/{epoch} Loss {loss.item()*scale:.4e}")
    opt_lang_model.eval()
    torch.save(opt_lang_model,'./models/model'+str(i)+'.pth')

Epoch 1/100 Loss 1.0072e+01: : 1it [00:00,  1.93it/s]

tensor(19331, device='cuda:0')
tensor(19690, device='cuda:0')
tensor(18622, device='cuda:0')


Epoch 1/100 Loss 9.8516e+00: : 7it [00:00, 12.45it/s]

tensor(19614, device='cuda:0')
tensor(19734, device='cuda:0')
tensor(19574, device='cuda:0')
tensor(19453, device='cuda:0')
tensor(19971, device='cuda:0')
tensor(19988, device='cuda:0')


Epoch 1/100 Loss 8.7792e+00: : 13it [00:01, 19.33it/s]

...

Epoch 1/100 Loss 8.2712e+00: : 444it [00:16, 25.35it/s]

tensor(19971, device='cuda:0')
tensor(18899, device='cuda:0')
tensor(19976, device='cuda:0')
tensor(19841, device='cuda:0')
tensor(19883, device='cuda:0')
tensor(19926, device='cuda:0')


Epoch 1/100 Loss 7.8151e+00: : 450it [00:17, 24.83it/s]

tensor(18679, device='cuda:0')
tensor(18538, device='cuda:0')
tensor(19421, device='cuda:0')
tensor(19670, device='cuda:0')
tensor(19695, device='cuda:0')
tensor(19734, device='cuda:0')


Epoch 1/100 Loss 6.8631e+00: : 456it [00:17, 25.92it/s]